In [1]:
import os
import pandas as pd
import numpy as np
import warnings
import hdbscan

from hdbscan import validity
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from collections import Counter
from itertools import combinations

warnings.filterwarnings('ignore')

## CHECK THIS BLOCK AND UPDATE VALUES

In [2]:
animes_path = '../data/anime.csv'
relations_path = '../data/anime_anime.csv'
clusters_path = '../data/review_clustering'
clusters_file_type = '.csv'
top_animes_path = '../data/top_animes.csv'
top_relations_path = '../data/top_relations.csv'


relations_weight = 0.5
genres_weight = 0.5

TITLES_TO_TAKE = 2000
rec_thresthold = 2

animes = pd.read_csv(animes_path)
relations = pd.read_csv(relations_path)

In [3]:
cluster_files = []
for filename in os.listdir(clusters_path):
    if filename.endswith(clusters_file_type):
        cluster_files.append(filename)


In [4]:
animes["score_count"]= pd.to_numeric(animes["score_count"], errors='raise', downcast='float')
top_animes = animes.sort_values("score_count", ascending=False).head(TITLES_TO_TAKE)

top_ids = top_animes['anime_id'].tolist()

top_relations = relations[relations['animeA'].isin(top_ids) & relations['animeB'].isin(top_ids)]
top_relations = top_relations[top_relations['num_recommenders'] > rec_thresthold]

top_animes.to_csv(top_animes_path, index=False)
top_relations.to_csv(top_relations_path, index=False)

In [5]:
def cluster_test(clusters, relations):
    # Step 1: Get cluster for animeA
    relations_clusterA = relations.merge(
        clusters, 
        left_on='animeA', 
        right_on='anime_id', 
        how='inner'
    ).rename(columns={'cluster': 'cluster_A'}).drop(columns=['anime_id'])

    # Step 2: Get cluster for animeB
    relations_both_clusters = relations_clusterA.merge(
        clusters, 
        left_on='animeB', 
        right_on='anime_id', 
        how='inner'
    ).rename(columns={'cluster': 'cluster_B'}).drop(columns=['anime_id'])

    # Step 3: Calculate statistics
    total_valid_relations = len(relations_both_clusters)
    same_cluster_count = (relations_both_clusters['cluster_A'] == relations_both_clusters['cluster_B']).sum()
    percentage = (same_cluster_count / total_valid_relations) * 100

    print(f"Original relations: {len(relations)}")
    print(f"Valid relations (both animes in clusters): {total_valid_relations}")
    print(f"Relations within same cluster: {same_cluster_count}")
    print(f"Percentage within same cluster: {percentage:.2f}%")

    return percentage
    

In [6]:
def genre_test(clusters, animes):
    # Create mappings
    anime_to_cluster = dict(zip(clusters['anime_id'], clusters['cluster']))
    anime_to_genres = {}

    # Parse genres for each anime
    for _, row in animes.iterrows():
        anime_id = row['anime_id']
        genres = set(row['genres'].split('|')) if pd.notna(row['genres']) else set()
        anime_to_genres[anime_id] = genres

    # Function to calculate Jaccard similarity
    def jaccard_similarity(set1, set2):
        if len(set1) == 0 and len(set2) == 0:
            return 1.0  # Both empty sets are considered identical
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        return intersection / union if union > 0 else 0.0

    # Group animes by cluster
    cluster_to_animes = {}
    for anime_id, cluster in anime_to_cluster.items():
        if anime_id in anime_to_genres:  # Only include animes that have genre data
            if cluster not in cluster_to_animes:
                cluster_to_animes[cluster] = []
            cluster_to_animes[cluster].append(anime_id)

    # Calculate average Jaccard similarity for each cluster
    cluster_similarities = {}

    for cluster, anime_list in cluster_to_animes.items():
        if len(anime_list) < 2:
            cluster_similarities[cluster] = None  # Can't calculate similarity with < 2 animes
            continue
        
        similarities = []
        for anime1, anime2 in combinations(anime_list, 2):
            genres1 = anime_to_genres[anime1]
            genres2 = anime_to_genres[anime2]
            similarity = jaccard_similarity(genres1, genres2)
            similarities.append(similarity)
        
        cluster_similarities[cluster] = sum(similarities) / len(similarities)

    # Display results
    # print("Average Jaccard similarity by cluster:")
    valid_clusters = {k: v for k, v in cluster_similarities.items() if v is not None}

    # for cluster, avg_similarity in valid_clusters.items():
    #     anime_count = len(cluster_to_animes[cluster])
    #     pair_count = len(list(combinations(cluster_to_animes[cluster], 2)))
    #     print(f"Cluster {cluster}: {avg_similarity:.4f} (from {pair_count} pairs, {anime_count} animes)")

    # Overall average across all clusters
    if valid_clusters:
        overall_average = sum(valid_clusters.values()) / len(valid_clusters) * 100
        print(f"Overall average Jaccard similarity: {overall_average:.2f}%")
        print(f"Calculated across {len(valid_clusters)} clusters")
    # else:
    #     print("No valid clusters found")

    return overall_average

In [7]:
def calculate_anime_clustering_fidelity(cluster_file_path, anime_file_path, relations_file_path, 
                                      content_weight=0.4, engagement_weight=0.3, 
                                      size_weight=0.2, metadata_weight=0.1):
    """
    Calculate clustering fidelity score for anime clusters.
    
    Parameters:
    -----------
    cluster_file_path : str
        Path to CSV with columns: anime_id, cluster
    anime_file_path : str 
        Path to anime.csv with metadata
    relations_file_path : str
        Path to anime relations CSV with recommendation data
    content_weight : float (default 0.4)
        Weight for content similarity component
    engagement_weight : float (default 0.3)
        Weight for user engagement component  
    size_weight : float (default 0.2)
        Weight for size balance component
    metadata_weight : float (default 0.1)
        Weight for metadata consistency component
        
    Returns:
    --------
    dict: Detailed breakdown of fidelity scores and final combined score
    """
    
    print("📊 Loading data files...")
    
    # Load data - PROPERLY parse CSV with quoted fields
    clusters_df = pd.read_csv(cluster_file_path)
    anime_df = pd.read_csv(anime_file_path, quotechar='"', escapechar='\\')  # Proper CSV parsing
    relations_df = pd.read_csv(relations_file_path)
    
    # Debug: Check if we're getting the right columns now
    print(f"📋 Anime columns found: {list(anime_df.columns)}")
    
    # Merge cluster data with anime metadata
    clustered_anime = clusters_df.merge(anime_df, left_on='anime_id', right_on='anime_id', how='left')
    
    print(f"✅ Loaded {len(clustered_anime)} anime across {clustered_anime['cluster'].nunique()} clusters")
    
    # ===== COMPONENT 1: CONTENT SIMILARITY (40%) =====
    #print("🎬 Calculating content similarity...")
    content_score = calculate_content_similarity(clustered_anime)
    
    # ===== COMPONENT 2: USER ENGAGEMENT (30%) =====
    #print("👥 Calculating user engagement...")
    engagement_score = calculate_user_engagement(clustered_anime, relations_df)
    
    # ===== COMPONENT 3: SIZE BALANCE (20%) =====
    #print("⚖️ Calculating size balance...")
    size_score = calculate_size_balance_simpsons(clustered_anime)
    
    # ===== COMPONENT 4: METADATA CONSISTENCY (10%) =====
    #print("📋 Calculating metadata consistency...")
    metadata_score = calculate_metadata_consistency(clustered_anime)
    
    # ===== FINAL COMBINED SCORE =====
    final_score = (content_weight * content_score + 
                   engagement_weight * engagement_score +
                   size_weight * size_score + 
                   metadata_weight * metadata_score)
    
    results = {
        'final_fidelity_score': final_score,
        'component_scores': {
            'content_similarity': content_score,
            'user_engagement': engagement_score, 
            'size_balance': size_score,
            'metadata_consistency': metadata_score
        },
        'component_weights': {
            'content_weight': content_weight,
            'engagement_weight': engagement_weight,
            'size_weight': size_weight, 
            'metadata_weight': metadata_weight
        },
        'cluster_stats': {
            'total_anime': len(clustered_anime),
            'num_clusters': clustered_anime['cluster'].nunique(),
            'avg_cluster_size': len(clustered_anime) / clustered_anime['cluster'].nunique(),
            'largest_cluster_size': clustered_anime['cluster'].value_counts().max(),
            'smallest_cluster_size': clustered_anime['cluster'].value_counts().min()
        }
    }
    
    #print(f"\n🎯 FINAL FIDELITY SCORE: {final_score:.3f}")
    return results

def calculate_content_similarity(clustered_anime):
    """Calculate content-based similarity within clusters using improved metrics."""
    
    # Verify imports are working
    try:
        from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
        print("   ✅ sklearn metrics imported successfully")
    except ImportError as e:
        print(f"   ❌ Import error: {e}")
        # Fallback to just silhouette
        return calculate_content_similarity_fallback(clustered_anime)
    
    # Create feature matrix from anime metadata
    features = []
    
    # Numerical features (normalized) - updated with correct column names
    numerical_cols = ['score', 'score_count', 'score_rank', 'popularity_rank', 
                     'members_count', 'favorites_count', 'num_episodes',
                     'watching_count', 'completed_count', 'on_hold_count', 
                     'dropped_count', 'plan_to_watch_count', 'total_count']
    
    for col in numerical_cols:
        if col in clustered_anime.columns:
            # Fill NaN and normalize
            values = pd.to_numeric(clustered_anime[col], errors='coerce').fillna(0)
            if values.std() > 0:  # Avoid division by zero
                normalized = (values - values.mean()) / values.std()
                features.append(normalized.values.reshape(-1, 1))
    
    # Categorical features (one-hot encoded) - updated with correct column names  
    categorical_cols = ['type', 'source_type', 'status', 'season']
    
    for col in categorical_cols:
        if col in clustered_anime.columns:
            # Simple one-hot encoding
            unique_vals = clustered_anime[col].fillna('Unknown').unique()
            for val in unique_vals:
                feature_col = (clustered_anime[col].fillna('Unknown') == val).astype(int)
                features.append(feature_col.values.reshape(-1, 1))
    
    # Genre similarity (Jaccard-based) - pipe-delimited
    if 'genres' in clustered_anime.columns:
        genre_features = calculate_genre_features(clustered_anime)
        if genre_features is not None:
            features.append(genre_features)
    
    # Studio similarity - pipe-delimited
    if 'studios' in clustered_anime.columns:
        studio_features = calculate_studio_features(clustered_anime)
        if studio_features is not None:
            features.append(studio_features)
    
    if not features:
        print("⚠️ No valid features found for content similarity")
        return 0.5  # Neutral score
    
    # Combine all features
    X = np.hstack(features)
    
    # Calculate improved clustering validity metrics
    try:
        labels = clustered_anime['cluster'].values
        if len(np.unique(labels)) < 2:
            return 0.0  # Single cluster gets 0 score
        
        # Method 1: Try DBCV (Density-Based Cluster Validation) - BEST
        dbcv_score = calculate_dbcv_score(X, labels)
        if dbcv_score is not None and not np.isnan(dbcv_score):
            print(f"   📊 Using DBCV score: {dbcv_score:.3f}")
            return max(0, dbcv_score)  # DBCV already in [0,1] range, higher = better
        
        # Method 2: Calinski-Harabasz Index (higher = better)
        try:
            ch_score = calinski_harabasz_score(X, labels)
            # Normalize CH score to roughly [0,1] range
            # Typical CH scores range from 10-1000+, so we use log normalization
            normalized_ch = min(1.0, np.log10(max(1, ch_score)) / 3.0)  # log10(1000) = 3
            print(f"   📊 Using Calinski-Harabasz score: {ch_score:.1f} → normalized: {normalized_ch:.3f}")
            return normalized_ch
        except Exception as e:
            print(f"   ⚠️ Calinski-Harabasz failed: {e}")
        
        # Method 3: Davies-Bouldin Index (lower = better, so we invert it)
        try:
            db_score = davies_bouldin_score(X, labels)
            # DB typically ranges from 0.5-3+, lower is better
            # Convert to [0,1] where higher = better
            normalized_db = max(0, 1 - (db_score / 3.0))
            print(f"   📊 Using Davies-Bouldin score: {db_score:.3f} → normalized: {normalized_db:.3f}")
            return normalized_db
        except Exception as e:
            print(f"   ⚠️ Davies-Bouldin failed: {e}")
        
        # Method 4: Fallback to Silhouette (worst option)
        try:
            sil_score = silhouette_score(X, labels)
            normalized_sil = (sil_score + 1) / 2  # Convert from [-1,1] to [0,1]
            print(f"   📊 Using Silhouette score: {sil_score:.3f} → normalized: {normalized_sil:.3f}")
            return normalized_sil
        except Exception as e:
            print(f"   ⚠️ All metrics failed: {e}")
            return 0.5
        
    except Exception as e:
        print(f"⚠️ Error calculating content similarity: {e}")
        return 0.5

def calculate_content_similarity_fallback(clustered_anime):
    """Fallback function with just silhouette score if imports fail."""
    print("   📉 Using fallback silhouette score only")
    
    # Simplified feature creation and silhouette calculation
    # [Same feature creation code as before but simpler]
    features = []
    
    # Just use numerical features for fallback
    numerical_cols = ['score', 'score_count', 'score_rank', 'popularity_rank', 
                     'members_count', 'favorites_count', 'num_episodes']
    
    for col in numerical_cols:
        if col in clustered_anime.columns:
            values = pd.to_numeric(clustered_anime[col], errors='coerce').fillna(0)
            if values.std() > 0:
                normalized = (values - values.mean()) / values.std()
                features.append(normalized.values.reshape(-1, 1))
    
    if not features:
        return 0.5
    
    X = np.hstack(features)
    labels = clustered_anime['cluster'].values
    
    try:
        sil_score = silhouette_score(X, labels)
        return (sil_score + 1) / 2
    except:
        return 0.5

def calculate_dbcv_score(X, labels):
    """Calculate DBCV (Density-Based Cluster Validation) score with better error handling."""
    try:
        import hdbscan
        from hdbscan import validity
        
        # DBCV requires dense matrix and specific label format
        if hasattr(X, 'toarray'):
            X_dense = X.toarray()
        else:
            X_dense = np.array(X)
        
        # Ensure we have valid data
        if X_dense.shape[0] < 2:
            print("   ⚠️ DBCV: Not enough data points")
            return None
            
        # Check for valid clusters (DBCV needs at least 2 clusters with 2+ points each)
        unique_labels, counts = np.unique(labels, return_counts=True)
        valid_clusters = counts >= 2  # Clusters with at least 2 points
        
        if np.sum(valid_clusters) < 2:
            print("   ⚠️ DBCV: Need at least 2 clusters with 2+ points each")
            return None
        
        # Filter to only valid clusters
        valid_cluster_ids = unique_labels[valid_clusters]
        valid_mask = np.isin(labels, valid_cluster_ids)
        
        if np.sum(valid_mask) < 4:  # Need at least 4 total points in valid clusters
            print("   ⚠️ DBCV: Not enough points in valid clusters")
            return None
        
        X_filtered = X_dense[valid_mask]
        labels_filtered = labels[valid_mask]
        
        # Remap labels to be consecutive starting from 0
        label_map = {old_label: new_label for new_label, old_label in enumerate(np.unique(labels_filtered))}
        mapped_labels = np.array([label_map[label] for label in labels_filtered])
        
        # Ensure proper data types
        X_final = X_filtered.astype(np.float64)
        labels_final = mapped_labels.astype(np.int32)
        
        # Additional safety checks
        if X_final.shape[1] == 0:
            print("   ⚠️ DBCV: No features in data")
            return None
            
        if np.any(np.isnan(X_final)) or np.any(np.isinf(X_final)):
            print("   ⚠️ DBCV: Data contains NaN or Inf values")
            return None
        
        # Calculate DBCV with error handling
        try:
            dbcv = validity.validity_index(X_final, labels_final)
            
            # Check if result is valid
            if dbcv is None or np.isnan(dbcv) or np.isinf(dbcv):
                print("   ⚠️ DBCV: Invalid result returned")
                return None
                
            print(f"   ✅ DBCV calculated successfully: {dbcv:.3f}")
            return dbcv
            
        except (ValueError, IndexError, RuntimeError) as e:
            print(f"   ⚠️ DBCV calculation error: {e}")
            return None
        
    except ImportError:
        print("   📦 HDBSCAN not installed. Install with: pip install hdbscan")
        return None
    except Exception as e:
        print(f"   ⚠️ DBCV setup failed: {e}")
        return None

def calculate_genre_features(clustered_anime):
    """Create genre feature matrix using multi-hot encoding."""
    try:
        # Get all unique genres (split on | not ,)
        all_genres = set()
        for genres_str in clustered_anime['genres'].dropna():
            if isinstance(genres_str, str):
                genres = [g.strip() for g in genres_str.split('|')]
                all_genres.update(genres)
        
        all_genres = sorted(list(all_genres))
        
        # Create multi-hot encoding
        genre_matrix = []
        for genres_str in clustered_anime['genres'].fillna(''):
            row = [0] * len(all_genres)
            if isinstance(genres_str, str) and genres_str:
                genres = [g.strip() for g in genres_str.split('|')]
                for genre in genres:
                    if genre in all_genres:
                        idx = all_genres.index(genre)
                        row[idx] = 1
            genre_matrix.append(row)
        
        return np.array(genre_matrix)
    except:
        return None

def calculate_studio_features(clustered_anime):
    """Create studio feature matrix."""
    try:
        # Get all unique studios (check if pipe-delimited too)
        all_studios = set()
        for studios_str in clustered_anime['studios'].dropna():
            if isinstance(studios_str, str):
                # Try pipe delimiter first, then comma fallback
                if '|' in studios_str:
                    studios = [s.strip() for s in studios_str.split('|')]
                else:
                    studios = [s.strip() for s in studios_str.split(',')]
                all_studios.update(studios)
        
        all_studios = sorted(list(all_studios))
        
        # Create multi-hot encoding
        studio_matrix = []
        for studios_str in clustered_anime['studios'].fillna(''):
            row = [0] * len(all_studios)
            if isinstance(studios_str, str) and studios_str:
                # Try pipe delimiter first, then comma fallback
                if '|' in studios_str:
                    studios = [s.strip() for s in studios_str.split('|')]
                else:
                    studios = [s.strip() for s in studios_str.split(',')]
                for studio in studios:
                    if studio in all_studios:
                        idx = all_studios.index(studio)
                        row[idx] = 1
            studio_matrix.append(row)
        
        return np.array(studio_matrix)
    except:
        return None

def calculate_user_engagement(clustered_anime, relations_df):
    """Calculate user engagement score using recommendation data."""
    
    try:
        # Filter for actual recommendations (not just relations)
        recommendations = relations_df[relations_df['recommendation'] == 1].copy()
        
        if len(recommendations) == 0:
            print("⚠️ No recommendation data found")
            return 0.5
        
        # Get anime in our clusters
        clustered_ids = set(clustered_anime['anime_id'].values)
        
        # Filter recommendations to only include clustered anime
        recs_filtered = recommendations[
            (recommendations['animeA'].isin(clustered_ids)) & 
            (recommendations['animeB'].isin(clustered_ids))
        ]
        
        if len(recs_filtered) == 0:
            print("⚠️ No recommendations found between clustered anime")
            return 0.5
        
        # Create cluster mapping
        cluster_map = dict(zip(clustered_anime['anime_id'], clustered_anime['cluster']))
        
        # Calculate same-cluster recommendation rate
        total_recommendations = 0
        same_cluster_recommendations = 0
        recommendation_strength_sum = 0
        
        for _, row in recs_filtered.iterrows():
            anime_a, anime_b = row['animeA'], row['animeB']
            num_recommenders = row.get('num_recommenders', 1)
            
            if anime_a in cluster_map and anime_b in cluster_map:
                total_recommendations += 1
                
                if cluster_map[anime_a] == cluster_map[anime_b]:
                    same_cluster_recommendations += 1
                    # Weight by recommendation strength
                    recommendation_strength_sum += num_recommenders
        
        if total_recommendations == 0:
            return 0.5
        
        # Base score: percentage of recommendations within same cluster
        base_score = same_cluster_recommendations / total_recommendations
        
        # Bonus for recommendation strength
        if same_cluster_recommendations > 0:
            avg_strength = recommendation_strength_sum / same_cluster_recommendations
            strength_bonus = min(0.1, avg_strength / 100)  # Max 10% bonus
            base_score += strength_bonus
        
        return min(1.0, base_score)  # Cap at 1.0
        
    except Exception as e:
        print(f"⚠️ Error calculating user engagement: {e}")
        return 0.5


def cluster_balance_score(df):
    """
    Calculate cluster balance score from dataframe with anime_id and cluster columns.
    
    Args:
        df: DataFrame with 'anime_id' and 'cluster' columns
    
    Returns:
        float: Balance score (0-1, higher = better balance)
    """
    cluster_sizes = df['cluster'].value_counts().values
    proportions = cluster_sizes / len(df)
    
    # Simpson's diversity
    simpson = 1 - np.sum(proportions**2)
    
    # Penalties
    cr1 = np.max(proportions)  # Largest cluster
    cr3 = np.sum(np.sort(proportions)[-3:])  # Top 3 clusters
    
    penalties = 0
    if cr1 > 0.25:
        penalties += (cr1 - 0.25) * 2
    if cr3 > 0.60:
        penalties += (cr3 - 0.60) * 1.5
    if len(cluster_sizes) < 4:
        penalties += (4 - len(cluster_sizes)) * 0.2
    
    return max(0, simpson - penalties)

def calculate_size_balance_simpsons(clustered_anime):
    """Calculate size balance score (penalizes giant clusters)."""
    
    cluster_counts = clustered_anime['cluster'].value_counts()
    
    # Method 1: Standard deviation penalty
    mean_size = cluster_counts.mean()
    std_size = cluster_counts.std()
    
    if mean_size == 0:
        return 0.0
    
    # Balance score: lower std relative to mean = better balance
    balance_score = max(0, 1 - (std_size / mean_size))
    
    # Method 2: Giant cluster penalty
    total_anime = len(clustered_anime)
    largest_cluster_pct = cluster_counts.max() / total_anime
    
    # Penalize if largest cluster > 40% of total
    giant_penalty = 0
    if largest_cluster_pct > 0.4:
        giant_penalty = (largest_cluster_pct - 0.4) * 2  # Linear penalty
    
    final_score = max(0, balance_score - giant_penalty)
    
    return final_score

def calculate_metadata_consistency(clustered_anime):
    """Calculate metadata consistency within clusters."""
    
    consistency_scores = []
    
    # Check consistency for each cluster
    for cluster_id in clustered_anime['cluster'].unique():
        cluster_data = clustered_anime[clustered_anime['cluster'] == cluster_id]
        
        if len(cluster_data) < 2:
            consistency_scores.append(1.0)  # Single anime clusters are perfectly consistent
            continue
        
        cluster_consistency = []
        
        # Type consistency (TV, Movie, etc.)
        if 'type' in cluster_data.columns:
            type_counts = cluster_data['type'].value_counts()
            type_consistency = type_counts.max() / len(cluster_data)
            cluster_consistency.append(type_consistency)
        
        # Season consistency (for seasonal anime)
        if 'season' in cluster_data.columns:
            season_counts = cluster_data['season'].dropna().value_counts()
            if len(season_counts) > 0:
                season_consistency = season_counts.max() / len(cluster_data.dropna(subset=['season']))
                cluster_consistency.append(season_consistency)
        
        # Studio consistency
        if 'studios' in cluster_data.columns:
            # For multi-studio anime, check if studios overlap
            studio_overlap = calculate_studio_overlap(cluster_data['studios'])
            cluster_consistency.append(studio_overlap)
        
        # Score range consistency (similar quality anime should cluster)
        if 'score' in cluster_data.columns:
            scores = cluster_data['score'].dropna()
            if len(scores) > 1:
                score_std = scores.std()
                # Normalize by overall score range (assume 1-10 scale)
                score_consistency = max(0, 1 - (score_std / 2.5))  # 2.5 = reasonable std for 10-point scale
                cluster_consistency.append(score_consistency)
        
        if cluster_consistency:
            consistency_scores.append(np.mean(cluster_consistency))
        else:
            consistency_scores.append(0.5)  # Neutral score if no consistency measures
    
    return np.mean(consistency_scores) if consistency_scores else 0.5

def calculate_studio_overlap(studio_series):
    """Calculate studio overlap within a cluster."""
    try:
        all_studios = []
        for studios_str in studio_series.dropna():
            if isinstance(studios_str, str):
                # Handle pipe delimiters for studios too
                if '|' in studios_str:
                    studios = [s.strip() for s in studios_str.split('|')]
                else:
                    studios = [s.strip() for s in studios_str.split(',')]
                all_studios.extend(studios)
        
        if not all_studios:
            return 0.5
        
        studio_counts = Counter(all_studios)
        most_common_count = studio_counts.most_common(1)[0][1] if studio_counts else 0
        
        # Consistency = how often the most common studio appears
        return most_common_count / len(studio_series.dropna()) if len(studio_series.dropna()) > 0 else 0.5
        
    except:
        return 0.5

In [8]:
max_score = 0
max_score_path = ""

for file in cluster_files:    
    cluster_file_path = os.path.join(clusters_path, file)

    results = calculate_anime_clustering_fidelity(
        cluster_file_path=cluster_file_path,
        anime_file_path=top_animes_path, 
        relations_file_path=top_relations_path,
        content_weight=0.4,
        engagement_weight=0.3,  # Reduced from 0.3
        size_weight=0.2,        # Increased from 0.2  
        metadata_weight=0.1
    )
    
    if results['final_fidelity_score'] > max_score:
        max_score = results['final_fidelity_score']
        max_score_path = cluster_file_path

    print("\n" + "="*50)
    print(f"📊 CLUSTERING FIDELITY RESULTS FOR {file}" )
    print("="*50)
    
    print(f"\n🎯 Final Fidelity Score: {results['final_fidelity_score']:.3f}")
    
    print(f"\n📋 Component Breakdown:")
    
    # Map component names to weight names
    component_weight_map = {
        'content_similarity': 'content_weight',
        'user_engagement': 'engagement_weight', 
        'size_balance': 'size_weight',
        'metadata_consistency': 'metadata_weight'
    }
    
    for component, score in results['component_scores'].items():
        weight_key = component_weight_map[component]
        weight = results['component_weights'][weight_key]
        contribution = weight * score
        print(f"   {component.replace('_', ' ').title()}: {score:.3f} (weight: {weight:.1f}, contribution: {contribution:.3f})")
    
    print(f"\n📊 Cluster Statistics:")
    for stat, value in results['cluster_stats'].items():
        print(f"   {stat.replace('_', ' ').title()}: {value}")
    
print("\n" + "="*50)
print(f"🏆 HIGHEST FIDELITY SCORE: {max_score:.3}")
print(f"   Path: {max_score_path}")
print("="*50)

📊 Loading data files...
📋 Anime columns found: ['anime_id', 'anime_url', 'title', 'synopsis', 'main_pic', 'type', 'source_type', 'num_episodes', 'status', 'start_date', 'end_date', 'season', 'studios', 'genres', 'score', 'score_count', 'score_rank', 'popularity_rank', 'members_count', 'favorites_count', 'watching_count', 'completed_count', 'on_hold_count', 'dropped_count', 'plan_to_watch_count', 'total_count', 'score_10_count', 'score_09_count', 'score_08_count', 'score_07_count', 'score_06_count', 'score_05_count', 'score_04_count', 'score_03_count', 'score_02_count', 'score_01_count', 'clubs', 'pics']
✅ Loaded 2000 anime across 10 clusters
   ✅ sklearn metrics imported successfully
   ✅ DBCV calculated successfully: -0.750
   📊 Using DBCV score: -0.750

📊 CLUSTERING FIDELITY RESULTS FOR hierarchical_10_clusters.csv

🎯 Final Fidelity Score: 0.368

📋 Component Breakdown:
   Content Similarity: 0.000 (weight: 0.4, contribution: 0.000)
   User Engagement: 1.000 (weight: 0.3, contribution